In [1]:
import sys
from pathlib import Path

# Allow `from src...` whether the kernel starts in the repo root or notebooks/.
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


# 02 — Feature Engineering & Multicollinearity

Build the leak-free feature table via `src.features.builder` and check VIF before finalising the feature set.

In [2]:
import pandas as pd
from src.models.data_split import load_processed_feature_table, FEATURE_COLS

feats = load_processed_feature_table()
feats[FEATURE_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
frequency,4323.0,6.466111,11.977051,1.000000,2.000000,3.000000,7.000000,280.000000
total_items_bought,4323.0,1800.941476,8018.987759,1.000000,213.500000,544.000000,1426.000000,296921.000000
unique_skus_bought,4323.0,83.730743,111.107579,1.000000,22.000000,49.000000,105.000000,2165.000000
monetary_total,4323.0,2892.132725,12779.904677,-200.780000,393.820000,947.610000,2348.385000,449288.570000
recency,4323.0,143.716899,114.859439,0.280556,37.587500,111.313194,268.501042,364.575694
customer_age_days,4323.0,403.947630,185.842201,0.392361,283.400347,428.589583,574.389931,638.677083
monetary_avg,4323.0,364.866098,488.273933,-200.780000,180.278485,278.606429,413.525833,13206.500000
avg_basket_size,4323.0,244.599982,1372.076694,1.000000,94.431818,155.666667,260.000000,87167.000000
interpurchase_interval_mean,4323.0,71.547180,81.928354,0.000000,7.825747,50.285833,97.998727,611.862500
interpurchase_interval_std,4323.0,35.998008,47.297882,0.000000,0.000000,20.402219,55.887651,440.492660


In [3]:
import numpy as np

X = feats[FEATURE_COLS].astype(float).replace([np.inf, -np.inf], 0).fillna(0)
Xv = X.to_numpy()
rows = []
for j, col in enumerate(X.columns):
    y = Xv[:, j]
    others = np.delete(Xv, j, axis=1)
    others = np.column_stack([np.ones(len(others)), others])
    coef, *_ = np.linalg.lstsq(others, y, rcond=None)
    resid = y - others @ coef
    ss_res = float(resid @ resid)
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot else 0.0
    rows.append((col, 1.0 / (1.0 - r2) if r2 < 1.0 else np.inf))

vif = pd.DataFrame(rows, columns=['feature', 'VIF'])
vif.sort_values('VIF', ascending=False).head(10)

,feature,VIF
16,engagement_score,9.279383
3,monetary_total,7.595696
1,total_items_bought,7.282195
4,recency,6.925103
5,customer_age_days,4.774466
0,frequency,3.139084
2,unique_skus_bought,2.197368
6,monetary_avg,1.841175
8,interpurchase_interval_mean,1.775900
7,avg_basket_size,1.737233


Features with VIF > 10 are flagged for review. `historical_clv` was removed because it duplicated `monetary_total` (perfect collinearity, plan D3).